
# Assignment 4 for Course 1MS041
Make sure you pass the `# ... Test` cells and
 submit your solution notebook in the corresponding assignment on the course website. You can submit multiple times before the deadline and your highest score will be used.

---
## Assignment 4, PROBLEM 1
Maximum Points = 24


    This time the assignment only consists of one problem, but we will do a more comprehensive analysis instead.

Consider the dataset `Corona_NLP_train.csv` that you can get from the course website [git](https://github.com/datascience-intro/1MS041-2024/blob/main/notebooks/data/Corona_NLP_train.csv). The data is "Coronavirus tweets NLP - Text Classification" that can be found on [kaggle](https://www.kaggle.com/datasets/datatattle/covid-19-nlp-text-classification). The data has several columns, but we will only be working with `OriginalTweet`and `Sentiment`.

1. [3p] Load the data and filter out those tweets that have `Sentiment`=`Neutral`. Let $X$ represent the `OriginalTweet` and let 
    $$
        Y = 
        \begin{cases}
        1 & \text{if sentiment is towards positive}
        \\
        0 & \text{if sentiment is towards negative}.
        \end{cases}
    $$
    Put the resulting arrays into the variables $X$ and $Y$. Split the data into three parts, train/test/validation where train is 60% of the data, test is 15% and validation is 25% of the data. Do not do this randomly, this is to make sure that we all did the same splits (we are in this case assuming the data is IID as presented in the dataset). That is [train,test,validation] is the splitting layout.

2. [4p] There are many ways to solve this classification problem. The first main issue to resolve is to convert the $X$ variable to something that you can feed into a machine learning model. For instance, you can first use [`CountVectorizer`](https://scikit-learn.org/1.5/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) as the first step. The step that comes after should be a `LogisticRegression` model, but for this to work you need to put together the `CountVectorizer` and the `LogisticRegression` model into a [`Pipeline`](https://scikit-learn.org/1.5/modules/generated/sklearn.pipeline.Pipeline.html#sklearn.pipeline.Pipeline). Fill in the variable `model` such that it accepts the raw text as input and outputs a number $0$ or $1$, make sure that `model.predict_proba` works for this. **Hint: You might need to play with the parameters of LogisticRegression to get convergence, make sure that it doesn't take too long or the autograder might kill your code**
3. [3p] Use your trained model and calculate the precision and recall on both classes. Fill in the corresponding variables with the answer.
4. [3p] Let us now define a cost function
    * A positive tweet that is classified as negative will have a cost of 1
    * A negative tweet that is classified as positive will have a cost of 5
    * Correct classifications cost 0
    
    complete filling the function `cost` to compute the cost of a prediction model under a certain prediction threshold (recall our precision recall lecture and the `predict_proba` function from trained models). 

5. [4p] Now, we wish to select the threshold of our classifier that minimizes the cost, fill in the selected threshold value in value `optimal_threshold`.
6. [4p] With your newly computed threshold value, compute the cost of putting this model in production by computing the cost using the validation data. Also provide a confidence interval of the cost using Hoeffdings inequality with a 99% confidence.
7. [3p] Let $t$ be the threshold you found and $f$ the model you fitted (one of the outputs of `predict_proba`), if we define the random variable
    $$
        C = (1-1_{f(X)\geq t})Y+5(1-Y)1_{f(X) \geq t}
    $$
    then $C$ denotes the cost of a randomly chosen tweet. In the previous step we estimated $\mathbb{E}[C]$ using the empirical mean. However, since the threshold is chosen to minimize cost it is likely that $C=0$ or $C=1$ than $C=5$ as such it will have a low variance. Compute the empirical variance of $C$ on the validation set. What would be the confidence interval if we used Bennett's inequality instead of Hoeffding in point 6 but with the computed empirical variance as our guess for the variance?

In [1]:

# Part 1

# Load the data from the file specified in the problem definition and make sure that it is loaded using
# the search path `data/Corona_NLP_train.csv`. This is to make sure the autograder and your computer have the same
# file path and can load the data correctly.

# Contrary to how many other problems are structured, this problem actually requires you to
# have X on the shape (n_samples, ) that is a 1-dimensional array. Otherwise it will cause a bunch
# of errors in the autograder or also in for instance CountVectorizer.

# Make sure that all your data is numpy arrays and not pandas dataframes or series.
import numpy as np
import pandas as pd

    #Load dataset
data = pd.read_csv('data/Corona_NLP_train.csv', delimiter=',', encoding='latin-1')
data = data[data['Sentiment']!= "Neutral"]
data['Sentiment'] = data['Sentiment'].str.contains("Positive").astype(int)
##print(data.info())


X = data["OriginalTweet"].to_numpy()
Y = data["Sentiment"].to_numpy()


n_samples = len(X)


#Inedex for slicing
n_train = int(0.60 * n_samples)
n_test = int(0.15 * n_samples)
n_valid = n_samples - n_train - n_test

X_train, Y_train = X[:n_train], Y[:n_train]
X_test, Y_test = X[n_train:n_train+n_test], Y[n_train:n_train+n_test]
X_valid, Y_valid = X[n_train + n_test:], Y[n_train + n_test:]

In [2]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
# Part 2

# Train a machine learning model or pipeline that can take the raw strings from X and predict Y=0,1 depending on the
# sentiment of the tweet. Store the trained model in the variable `model`.
pipe = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('classifier', LogisticRegression(max_iter=1000))
    ])



model = pipe.fit(X_train, Y_train)
test_res_probs =  model.predict_proba(X_test)

In [3]:

# Part 3
# Evaluate the model on the test set and calculate precision, and recall on both classes. Store the results in the
# variables `precision_0`, `precision_1`, `recall_0`, `recall_1`.


from sklearn.metrics import precision_score, recall_score
y_pred = model.predict(X_test)

#HEre we do precision for both classes using 
##Precision: TP / (TP+FP) 
precision_0 = precision_score(Y_test, y_pred, pos_label=0)
precision_1 = precision_score(Y_test, y_pred, pos_label=1)

##Recall : TP / (TP+FN)
recall_0 = recall_score(Y_test, y_pred, pos_label=0)
recall_1 = recall_score(Y_test, y_pred, pos_label=1)

In [4]:
# Part 4
print(model.predict_proba(X)) ## Here we need to select 

def cost(model,threshold,X,Y):
    # Hint, make sure that the model has a predict_proba method
    # think about how the decision is made based on the probabilities
    # and how the threshold can be used to make the decision.
    # For reference take a look at the lecture notes "Bayes classifier"
    # which contains how the decision is made based on the probabilities when the threshold is 0.5.
    
    # Fill in what is missing to compute the cost and return it
    # Note that we are interested in average cost
    probs = model.predict_proba(X)
    preds = (probs[:,1] >= threshold).astype(int)

    cost_vec = []
    for i in range(len(preds)):
        pred_val = preds[i]      
        true_val = Y[i]          
    
        if pred_val == 0 and true_val == 1:
            cost_vec.append(1)   
        elif pred_val == 1 and true_val == 0:
            cost_vec.append(5)   
        else:
            cost_vec.append(0)  
    cost_avg = sum(cost_vec)/len(cost_vec)
    return cost_avg  

[[0.07769194 0.92230806]
 [0.05165099 0.94834901]
 [0.06115848 0.93884152]
 ...
 [0.95953888 0.04046112]
 [0.08088465 0.91911535]
 [0.87534831 0.12465169]]


In [5]:
# Part 5
# Find the optimal threshold for the model on the test set. Store the threshold in the variable `optimal_threshold`
# and the cost at the optimal threshold in the variable `cost_at_optimal_threshold` evaluated on the test set.
thresholds = np.linspace(0,1,200)
costs = [cost(model, t, X_test, Y_test) for t in thresholds]

minste_index = np.argmin(costs)

optimal_threshold = thresholds[minste_index]

cost_at_optimal_threshold = costs[minste_index]
print("Optimal threshold",optimal_threshold)
print("Cost at optimal: ",cost_at_optimal_threshold)

Optimal threshold 0.7839195979899498
Cost at optimal:  0.27392344497607657


In [6]:
# Part 6

import math as m
cost_at_optimal_threshold_valid = cost(model, optimal_threshold, X_valid, Y_valid)

epsilon = m.sqrt(((5)**2 * m.log(2 / 0.01)) / (2 * len(Y_valid)))

lower_bound = cost_at_optimal_threshold_valid - epsilon
upper_bound = cost_at_optimal_threshold_valid + epsilon

cost_interval_valid = (lower_bound, upper_bound)
print(cost_interval_valid)

assert(type(cost_interval_valid) == tuple)
assert(len(cost_interval_valid) == 2)

(0.18928703086625748, 0.3672783841062371)


In [ ]:
# Part 7
probs = model.predict_proba(X_valid)
preds = (probs[:,1] >= optimal_threshold).astype(int)

cost_vec2 = []
for i in range(len(preds)):
    pred_val = preds[i]      
    true_val = Y_valid[i]          

    if pred_val == 0 and true_val == 1:
        cost_vec2.append(1)   
    elif pred_val == 1 and true_val == 0:
        cost_vec2.append(5)   
    else:
        cost_vec2.append(0)  

variance_of_C = np.var(cost_vec2)


###Second part,  makong the interval C using bennets inequality
def h(u):
    h = (1 + u) * np.log(1 + u) - u
    return h

b = 5
CI_lim = 0.01

mal = (b**2 * np.log(2/CI_lim)) / (len(Y_valid)*variance_of_C)
u_verdi = np.linspace(0.00001, 1000, 1000000)
hverdi = np.array([h(u) for u in u_verdi])

laveste_feil = np.argmin(np.abs(hverdi - mal))
losning = u_verdi[laveste_feil]

e_bennet = (variance_of_C/b) * losning


lower_bound =cost_at_optimal_threshold_valid - e_bennet 
upper_bound = cost_at_optimal_threshold_valid + e_bennet


interval_of_C = (lower_bound, upper_bound)
print(interval_of_C)
assert(type(interval_of_C) == tuple)
assert(len(interval_of_C) == 2)


(np.float64(0.2473126583434126), np.float64(0.30925275662908197))
